# Replication notebook
This notebook walks through the package step by step: data loading, feature construction, sample splitting, model estimation, and evaluation.

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

import logging
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
import matplotlib.pyplot as plt

from config import (
    CacheConfig,
    TimeframeConfig,
    DataFilesConfig,
    RunControlConfig,
    SplitConfig,
    DataRegimeConfig,
    HyperGridConfig,
    ReproducibilityConfig,
    CharacteristicsFrequency,
    LoggingConfig,
    ExpandingWindowConfig,
    ModelSelectionConfig,
)

from io_utils import ensure_dir, save_parquet, set_global_seed, setup_project_logger
from data_inputs import load_datashare, load_crsp_monthly, load_macro_monthly
from dataset_builder import impute_characteristics_by_month_cross_sectional_median, compute_missingness_for_characteristics, save_missingness_comparison_plot, summarize_columns, save_missingness_three_comparison_plot


In [ ]:
repro_cfg = ReproducibilityConfig(random_state=42)
cache_cfg = CacheConfig()
tf_cfg = TimeframeConfig()
data_cfg = DataFilesConfig()
run_ctrl_cfg = RunControlConfig()
split_cfg = SplitConfig()
regime_cfg = DataRegimeConfig(mode="full")
grid_cfg = HyperGridConfig(use_extended_grids=False)
freq_cfg = CharacteristicsFrequency()
log_cfg = LoggingConfig()
expand_cfg = ExpandingWindowConfig()
model_cfg = ModelSelectionConfig()


In [ ]:

# Set global seed ONCE at the start - ensures reproducibility across all runs
set_global_seed(repro_cfg.random_state, repro_cfg.torch_deterministic)

logger, log_path = setup_project_logger(
    logger_name=log_cfg.logger_name,
    log_dir=log_cfg.log_dir,
    regime_mode=regime_cfg.mode,
    overwrite_log=log_cfg.overwrite_log,
    file_level_full=log_cfg.file_level_full,
    file_level_coding=log_cfg.file_level_coding,
    console_level_full=log_cfg.console_level_full,
    console_level_coding=log_cfg.console_level_coding,
)

logger.info("Starting experiment run.")
logger.info(f"Random seed: {repro_cfg.random_state}")
logger.info(f"Cache dir: {cache_cfg.cache_dir}")
logger.info(f"Data regime: {regime_cfg.mode}")


In [ ]:

# Log model selection configuration
if model_cfg.run_all_models:
    logger.info("Model selection: Running ALL models (including LSTM)")
    models_to_run = None  # None signals to run all models
elif model_cfg.models_to_run is not None:
    logger.info(f"Model selection: Running custom subset: {model_cfg.models_to_run}")
    models_to_run = model_cfg.models_to_run
else:
    logger.info("Model selection: Running default (all models)")
    models_to_run = None

ensure_dir("output")
ensure_dir(cache_cfg.cache_dir)


In [ ]:
complete_dataset_path=f"{cache_cfg.cache_dir}/complete_dataset.parquet"
descriptives_path=cache_cfg.descriptives_dir

feature_panel_path = f"{cache_cfg.cache_dir}/feature_panel_{regime_cfg.mode}.parquet"
feature_cols_path = f"{cache_cfg.cache_dir}/feature_cols_{regime_cfg.mode}.pkl"

In [ ]:
datashare_path=data_cfg.datashare_path
crsp_path=data_cfg.crsp_monthly_path
macro_path=data_cfg.macro_path
out_path=complete_dataset_path
descriptives_path=descriptives_path
cache_enabled=cache_cfg.enabled
possible_crsp_cols=data_cfg.possible_crsp_cols
possible_marco_cols=data_cfg.possible_marco_cols
cols_chara=data_cfg.chara_cols
cols_vars_monthly=freq_cfg.cols_vars_monthly
cols_vars_quarterly=freq_cfg.cols_vars_quarterly
cols_vars_annual=freq_cfg.cols_vars_annual

In [ ]:
logger.info(f"Building complete dataset, output: {out_path}")


logger.debug("Loading source datasets")

crsp = load_crsp_monthly(crsp_path, possible_crsp_cols)
macro = load_macro_monthly(macro_path, possible_marco_cols)
ds = load_datashare(datashare_path)

#-------------
date_1987_05 = pd.Period("1987-05", freq="M")
#-------------
#-------------
test_1 = ds.loc[ds["date"] == date_1987_05].copy()
#-------------

# Columns 3-96 in datashare.csv = 94 characteristics.
characteristic_cols = cols_chara
logger.debug(f"Identified {len(characteristic_cols)} characteristic columns")

logger.debug("Merging datashare with CRSP")
merged = ds.merge(
    crsp[["permno", "date", "ret", "dlret"]],
    on=["permno", "date"],
    how="inner",
    # validate="one_to_one",
)

#-------------
test_2 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

logger.debug("Merging with macro data")
merged = merged.merge(macro, on="date", how="left")

merged = merged.sort_values(["permno", "date"]).reset_index(drop=True)


#-------------
test_3 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# ------------------------------------------------------------------
# ret_total
# ------------------------------------------------------------------
logger.debug("Creating ret_total")

has_return_data = (merged["ret"].notna() | merged["dlret"].notna())

merged["ret_total"] = (
    (1.0 + merged["ret"].fillna(0.0))
    * (1.0 + merged["dlret"].fillna(0.0))
    - 1.0
).where(has_return_data)    

#-------------
test_4 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# ------------------------------------------------------------------
# Missingness visualisation and imputation
# ------------------------------------------------------------------
cols_chara_and_ret_total = characteristic_cols + ["ret_total"]

# Visualisation before imputation
missing_before = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

before_csv = f"{descriptives_path}/charas_missingness_before.csv"
missing_before.to_csv(before_csv, index=False)

#-------------
test_5 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------


logger.debug(f"Saved missingness before imputation: {before_csv}")

# Imputation
logger.info("Imputing missing characteristics using monthly cross-sectional medians")
merged = impute_characteristics_by_month_cross_sectional_median(merged, cols_chara_and_ret_total)

#-------------
test_5 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# Visualisation after imputation
missing_after = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_csv = f"{descriptives_path}/charas_missingness_after.csv"
missing_after.to_csv(after_csv, index=False)


logger.debug(f"Saved missingness after imputation: {after_csv}")

# Comparison plot
comparison_jpg = f"{descriptives_path}/charas_missingness_comparison.jpg"
save_missingness_comparison_plot(
    missing_before,
    missing_after,
    comparison_jpg,
    title="Missing Data Percentage: Before vs After Imputation",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")


# ------------------------------------------------------------------
# Temporal shifts 
# ------------------------------------------------------------------


#-------------
test_6 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# Build next-month excess return target.
logger.debug("Building lead return target (shift ret_total)")

merged = merged.sort_values(["permno", "date"])
merged["ret_total"] = merged.groupby("permno")["ret_total"].shift(-1)


# Build shifts for monthly, quarterly and annual characteristcs
logger.debug("Building characteristic shifts")
merged = merged.sort_values(["permno", "date"])
grouped = merged.groupby("permno")

for i in merged.columns:
    if i in cols_vars_monthly:
        merged[i] = grouped[i].shift(-1)
    elif i in cols_vars_quarterly:
        merged[i] = grouped[i].shift(-3)
    elif i in cols_vars_annual:
        merged[i] = grouped[i].shift(-6)


#-------------
test_7 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# ------------------------------------------------------------------
# Summary 
# ------------------------------------------------------------------
summary_before = summarize_columns(merged, cols_chara_and_ret_total)

summary_before_csv = f"{descriptives_path}/summary_before.csv"
summary_before.to_csv(summary_before_csv, index=False)

# Define inclusive monthly boundaries
start_period = pd.Period("1957-10", freq="M")
end_period = pd.Period("2021-06", freq="M")

# Keep observations from October 1957 through June 2021

mask = (
    (merged["date"] >= start_period)
    & (merged["date"] <= end_period)
)
merged = merged.loc[mask].copy()

logger.debug(f"Dataset reduced due to missing values to : {merged["date"].min()} and {merged["date"].max()}")


#-------------
test_8 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

# Visualisation of missingness after temporal cuts
missing_after_cut = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_cut_csv = f"{descriptives_path}/charas_missingness_after_cut.csv"
missing_after_cut.to_csv(after_cut_csv, index=False)

logger.debug(f"Saved missingness after temporal cut: {after_cut_csv}")

# Comparison plot
comparison_three_jpg = f"{descriptives_path}/charas_missingness_comparison_three.jpg"
save_missingness_three_comparison_plot(
    missing_before,
    missing_after,
    missing_after_cut,
    comparison_three_jpg,
    title="Missing Data Percentage: Before vs After Imputation vs After Temporal Reduction",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")


#-------------
test_9 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------
#-------------
test_1_csv = f"{descriptives_path}/test_1.csv"
test_1.to_csv(test_1_csv, index=False)

test_2_csv = f"{descriptives_path}/test_2.csv"
test_2.to_csv(test_2_csv, index=False)

test_3_csv = f"{descriptives_path}/test_3.csv"
test_3.to_csv(test_3_csv, index=False)

test_4_csv = f"{descriptives_path}/test_4.csv"
test_4.to_csv(test_4_csv, index=False)

test_5_csv = f"{descriptives_path}/test_5.csv"
test_5.to_csv(test_5_csv, index=False)

test_6_csv = f"{descriptives_path}/test_6.csv"
test_6.to_csv(test_6_csv, index=False)

test_7_csv = f"{descriptives_path}/test_7.csv"
test_7.to_csv(test_7_csv, index=False)

test_8_csv = f"{descriptives_path}/test_8.csv"
test_8.to_csv(test_8_csv, index=False)

test_9_csv = f"{descriptives_path}/test_9.csv"
test_9.to_csv(test_9_csv, index=False)
#-------------



logger.info(f"Complete dataset built: {merged.shape}")
save_parquet(merged, out_path, enabled=cache_enabled)